# 05 · 施設間シフトの特徴空間分解（実験 A）

ドメインシフトが**特徴空間のどこに入っているか**を、エンコーダごとに分解する。
現行アーキテクチャは late fusion で融合後の単一表現が無いので、5 つのエンコーダ
（T1 / T2 / FLAIR / T1c / unified）それぞれの CLS 特徴（ヘッド直前, 384 次元）について

$$\Delta = \overline{f_{\text{test}}} - \overline{f_{\text{val}}} \in \mathbb{R}^{384}$$

を、evidence ヘッド $\mathrm{Linear}(384 \to 2)$ の機能的部分空間 $\mathrm{span}\{W_1, W_0\}$ に射影する：

- $u_p = (W_1 - W_0)/\|W_1 - W_0\|$ … 判別方向（$\hat p$ を動かす）
- $u_s = \mathrm{orth}(W_1 + W_0; u_p)$ を正規化 … 確信方向（$S$ を動かす）
- $\Delta_p = \langle\Delta, u_p\rangle$（criterion シフト）, $\Delta_s = \langle\Delta, u_s\rangle$（evidence 総量シフト）,
  $\Delta_{\text{null}} = \|\Delta - \Delta_p u_p - \Delta_s u_s\|$（ヘッドから不可視な移動）

**スケール正規化**：384 次元では直交成分のノルムが自明に大きい。$d_p = \Delta_p/\sigma_p$, $d_s = \Delta_s/\sigma_s$
（$\sigma$ は val 特徴の各方向への射影の標準偏差）、$\Delta_{\text{null}}$ は「val を半分に割った null Δ の残差ノルム」
（ブートストラップ 100 回）を雑音床としてその何倍かで報告する。雑音床は raw（仕様どおり、val の半分同士なので保守的）と
size-matched（$\sqrt{(1/n_{\text{val}} + 1/n_{\text{test}})/(2/\text{half})}$ を掛けて実際の Δ と同じ標本サイズに揃えたもの）の両方を出す。

**機能への翻訳**（一次近似、softplus の局所勾配は無視）：エンコーダ $i$ の融合ロジットへの寄与シフト $\delta z_i \approx \langle\Delta, W_1 - W_0\rangle$、
evidence 総量シフト $\approx \langle\Delta, W_1 + W_0\rangle$。特徴を保存してあるので、softplus を通した**厳密な**ヘッド出力の変化も併記する。

ロジックは `openidh_model/eval/{features,shift_decomp,shift_report}.py`。特徴は `scripts/dump_features.py`
（HPC: `qsub qsub/dump_features.sh`）が `runs/<run>/features_{val,test}.npz` + `head_weights.npz` に保存済みで、
この notebook は**推論なし**で再実行できる。ヘッドの行対応（row 0 = e1/Mut, row 1 = e0/WT）は dump 時に全症例で
手計算 evidence と照合し、さらに 1 症例の融合 α/β を predictions.csv と照合してある（`features_meta.json`）。

In [ ]:
import os, sys, json
from pathlib import Path
_root = Path.cwd()
while not (_root / "data" / "metadata").exists():
    _root = _root.parent
os.chdir(_root); sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from openidh_model.eval.features import ENCODERS
from openidh_model.eval import shift_report as sr

In [ ]:
# Parameters (papermill injects overrides here).
runs_dir    = "runs"
units       = ["foldA", "foldB", "field", "vendor"]   # 4 units x seeds = the 12 shift runs
seeds       = [0, 1, 2]
control_run = "rand0_s0"                             # in-distribution negative control (1 run)
n_boot      = 100                                    # half-split draws for the noise floor
boot_seed   = 0
out_json    = "results/shift_decomposition.json"
out_md      = "results/shift_decomposition.md"
fig_path    = "results/figures/shift_decomposition_heatmap.png"

## 0. 対象ランと検算ログ

各ランの `features_meta.json`（dump 時のヘッド再計算誤差と、predictions.csv との融合 α/β 照合）を確認する。
特徴の無いランはここで明示し、以降の解析はあるものだけで進める。

In [ ]:
runs = [f"{u}_s{s}" for u in units for s in seeds] + ([control_run] if control_run else [])
present = [r for r in runs if sr.load_run(runs_dir, r) is not None]
missing = [r for r in runs if r not in present]
print(f"{len(present)}/{len(runs)} runs have feature dumps")
if missing:
    print("MISSING (no features_*.npz / head_weights.npz — run scripts/dump_features.py where the checkpoints are):")
    print("  " + " ".join(missing))
rows = []
for r in present:
    m = json.loads((Path(runs_dir) / r / "features_meta.json").read_text())
    for role in ("val", "test"):
        c = m["checks"][role]
        rows.append({"run": r, "role": role, "n": m[f"n_{role}"],
                     "head_max_abs_err": max(c["head_max_abs_err"].values()), "fused_check": c["fused"]})
checks = pd.DataFrame(rows)
assert (checks["head_max_abs_err"] < 1e-4).all(), "head recomputation mismatch — row order?"
display(checks)

## 1. 分解表：ラン × エンコーダ × {d_p, d_s, Δ_null/雑音床}

3 seed の平均 ± sd。`se_d` は標準化平均差の解析的標準誤差 $\sqrt{1/n_{\text{val}}+1/n_{\text{test}}}$（d_p, d_s の雑音の目安）。

In [ ]:
df, _ = sr.decompose_all(runs_dir, present, n_boot=n_boot, seed=boot_seed)
summ = sr.summarize_units(df)
per_run_cols = ["run", "unit", "enc", "n_val", "n_test", "d_p", "d_s", "se_d", "null_over_floor",
                "null_over_floor_matched", "dz_lin", "dS_lin", "dS_exact", "dS_exact_rel", "cos_sum_diff"]
display(df[per_run_cols].round(3))
print(sr.table_markdown(summ))

In [ ]:
fig = sr.heatmap(summ, fig_path, title="Feature-space shift per encoder (mean ± sd over seeds)")
plt.show()

## 2. ユニット間比較（本丸）

- **field / vendor** は train・val・test がすべて UTSW で有病率シフトがほぼゼロの「1b プローブ」。ここでの Δ の入り方
  （d_p に入るか d_s に入るか）が撮像由来シフトの正体。
- **LOSO-A / LOSO-B** は 1a（症例構成）+ 1b（撮像）の混合。field/vendor のパターンとの差分が 1a の寄与の示唆。
- **random f0** は分布内の陰性対照：全成分が雑音床近傍のはず。

判定はユニットごとに、各成分を自分の雑音床（d_p, d_s は se_d、Δ_null は size-matched 雑音床）で割った比を
エンコーダ平均し、2 倍を超えるものを「動いた」とみなす（`sr.judge_unit`）。

In [ ]:
unit_labels = [u for u in sr.UNIT_ORDER if u in set(summ["unit"])] + \
              [u for u in summ["unit"].unique() if u not in sr.UNIT_ORDER]
verdicts = {u: sr.judge_unit(summ, u) for u in unit_labels}
vt = pd.DataFrame([{"unit": u, **{f"{k}/floor": round(v, 2) for k, v in d["ratios"].items()},
                    "verdict": d["verdict"]} for u, d in verdicts.items()])
display(vt)

# per-unit, per-encoder view of where the mass is: |d_p| vs |d_s| (both in val-σ units)
piv = summ.pivot(index="enc", columns="unit", values=["d_p_mean", "d_s_mean", "null_over_floor_matched_mean"]).round(2)
display(piv)

## 3. 層別 Δ による 1a / 1b の直接分離（LOSO のみ）

(grade 二値 × 年齢 3 区分) の層ごとに Δ を計算し、Kitagawa 分解

$$\Delta = \underbrace{\sum_k \pi^{\text{test}}_k(\mu^{\text{test}}_k - \mu^{\text{val}}_k)}_{\text{層内 } \approx \text{撮像由来 (1b)}}
+ \underbrace{\sum_k (\pi^{\text{test}}_k - \pi^{\text{val}}_k)\,\mu^{\text{val}}_k}_{\text{症例構成由来 (1a)}}$$

で、全体 Δ − 症例数（test 側）加重平均の層内 Δ ≈ 症例構成由来の成分、とする。ラベルは使わない（age / grade のみ、
既存の較正解析と同じ共変量）。層は両側に 3 例以上あるものだけを使い、test 側のカバー率を併記する。
UPenn は GBM コホートなので grade 4 固定（データセット定義）。各部分を $(u_p, u_s)$ に射影し val-σ 単位で示す。

In [ ]:
loso_runs = [r for r in present if sr.unit_of(r)[1] == "site"]
strat_rows = []
for r in loso_runs:
    rd = sr.load_run(runs_dir, r)
    strat_rows += sr.stratified_run(rd)
strat = pd.DataFrame(strat_rows)
if len(strat):
    strat["enc"] = pd.Categorical(strat["enc"], ENCODERS)
    for r in loso_runs[:1]:
        print(f"strata counts, {r}:"); display(sr.strata_table(sr.load_run(runs_dir, r)))
    cols = ["delta_dp", "within_dp", "composition_dp", "delta_ds", "within_ds", "composition_ds", "covered"]
    g = strat.groupby(["unit", "enc"], observed=True)[cols]
    strat_summ = pd.concat([g.mean().add_suffix("_mean"), g.std(ddof=1).fillna(0).add_suffix("_sd")], axis=1).round(2)
    display(strat_summ)
    # share of the in-plane shift that survives holding the case mix fixed
    for u in strat["unit"].unique():
        s = strat[strat.unit == u]
        for comp in ("dp", "ds"):
            tot = s[f"delta_{comp}"].abs().mean(); win = s[f"within_{comp}"].abs().mean()
            print(f"{u:8s} {comp}: |within| / |delta| = {win / tot if tot else float('nan'):.2f}  "
                  f"(mean over encoders x seeds; covered {s['covered'].mean():.2f})")
else:
    print("no LOSO run has features — analysis 3 skipped")

## 4. vendor の evidence 崩壊の帰属

予想：vendor ランで T1 / T2 / FLAIR の d_s が大きく負、T1c は小さい。unified はどちら側か。
線形近似 $\langle\Delta, W_1+W_0\rangle$ と、特徴から softplus を通して計算した**厳密な** raw evidence 変化
（`dS_exact_rel` = test/val − 1）、および predictions.csv の**スケール後** evidence 変化（既報の「半減」）を並べる。

In [ ]:
rows = []
for r in present:
    rd = sr.load_run(runs_dir, r)
    ev = sr.evidence_change_from_predictions(rd)
    for enc in ENCODERS + ["tabular"]:
        if enc in ev:
            rows.append({"run": r, "unit": sr.unit_of(r)[0], "enc": enc, **{f"scaled_{k}": v for k, v in ev[enc].items()}})
scaled = pd.DataFrame(rows)
scaled["enc"] = pd.Categorical(scaled["enc"], ENCODERS + ["tabular"])
a4 = df[["run", "unit", "enc", "d_s", "dS_lin", "dS_exact_rel", "S_val_mean", "S_test_mean"]].astype({"enc": str}).merge(
    scaled.astype({"enc": str}), on=["run", "unit", "enc"], how="left")
a4["enc"] = pd.Categorical(a4["enc"], ENCODERS)
a4_summ = a4.groupby(["unit", "enc"], observed=True)[["d_s", "dS_lin", "dS_exact_rel", "scaled_rel"]].agg(["mean", "std"]).round(3)
display(a4_summ)
# order encoders by mean d_s in the vendor unit, if present
if "vendor Philips" in set(a4["unit"]):
    v = a4[a4.unit == "vendor Philips"].groupby("enc", observed=True)[["d_s", "dS_exact_rel", "scaled_rel"]].mean().sort_values("d_s")
    print("vendor Philips — encoders ordered by d_s:"); display(v.round(3))

## 5. 陰性対照

1. **random f0（分布内）**：同じ分解で全成分が雑音床近傍のはず（§1 の表と図の最終列、§2 の verdict）。
2. **検算**：$\sum_i \langle\Delta_i, W_1-W_0\rangle$（エンコーダ別ロジットシフトの一次近似の合計；体積スケール・(+1) 補正は無視）
   の符号が、実測の融合ロジット変化（predictions.csv: test 平均 − val 平均）および較正解析のオフセット
   （calibration.json `prior_offset`、LOSO-B で正）と整合するか。

In [ ]:
rows = []
for r in present:
    rd = sr.load_run(runs_dir, r)
    dec = [x for x in df.to_dict("records") if x["run"] == r]
    rows.append(sr.logit_shift_check(rd, dec))
lc = pd.DataFrame(rows).drop(columns=["dz_lin_by_enc"])
display(lc.round(3))
if "random f0" in set(summ["unit"]):
    print("random f0 verdict:", verdicts["random f0"]["verdict"])
for u in lc["unit"].unique():
    s = lc[lc.unit == u]
    sgn = lambda x: "+" if x > 0 else "−"
    cols = [c for c in ("fused_logit_shift", "prior_offset") if c in s]
    print(f"{u:14s} Σδz_lin {s['dz_lin_sum'].mean():+.2f}  Σδz_exact {s['dz_exact_sum'].mean():+.2f}  " +
          "  ".join(f"{c} {s[c].mean():+.2f}" for c in cols) +
          "  | signs: " + " ".join(sgn(s[c].mean()) for c in ["dz_lin_sum"] + cols))

## 6. 判定と出力

判定基準（§2 への翻訳）：
- field/vendor で **d_s が支配的（特に負）** → 1b は主に evidence 総量を壊す → エンコーダ側介入（アフィン不変拡張・関係特徴）に投資する価値が確定
- field/vendor で **d_p が支配的** → 撮像差も criterion を動かす（切片補正の残差の源）→ エンコーダ介入は較正にも効くはず
- **全成分が雑音床近傍** → 特徴は動いていない → 介入対象はヘッド側。その旨を明記して止まる
- **Δ_null が支配的** → 表現は動くがヘッドが無視している → 現状無害、エンコーダ不安定性の記録

In [ ]:
out = {
    "runs": present, "missing": missing, "n_boot": n_boot,
    "per_run": df.drop(columns=[c for c in df.columns if c.startswith("floor_null_p95")]).to_dict("records"),
    "summary": summ.to_dict("records"),
    "verdicts": verdicts,
    "stratified": strat.drop(columns=["age_cuts"]).to_dict("records") if len(strat) else [],
    "stratified_age_cuts": {r["run"]: list(map(float, r["age_cuts"])) for r in strat_rows},
    "evidence_change": a4.to_dict("records"),
    "logit_check": lc.to_dict("records"),
}
Path(out_json).parent.mkdir(parents=True, exist_ok=True)
def _default(o):
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, np.ndarray): return o.tolist()
    if hasattr(o, "item"): return o.item()
    return str(o)
Path(out_json).write_text(json.dumps(out, indent=1, default=_default, ensure_ascii=False))

md = ["## 実験 A：施設間シフトの特徴空間分解", "",
      f"対象 {len(present)} ラン" + (f"（欠損: {' '.join(missing)}）" if missing else "") + f"、雑音床 = val 半分割 × {n_boot}", "",
      "### 1. 分解表（3 seed 平均 ± sd）", sr.table_markdown(summ), "",
      "### 2. ユニット判定", sr.df_markdown(vt), ""]
if len(strat):
    md += ["### 3. 層別 Δ（LOSO、val-σ 単位、within = 撮像由来 / composition = 症例構成由来）",
           sr.df_markdown(strat_summ.reset_index()), ""]
md += ["### 4. evidence 総量変化（d_s / 線形 ⟨Δ,W1+W0⟩ / 厳密 raw / スケール後 predictions.csv）",
       sr.df_markdown(a4_summ.reset_index(), 3), "",
       "### 5. 検算（Σδz の符号 vs 実測ロジット変化 vs prior offset）", sr.df_markdown(lc, 3), ""]
Path(out_md).write_text("\n".join(md))
print(f"wrote {out_json}, {out_md}, {fig_path}")